# 🗄️ SQL Analysis — World Cup 2026 Database (MySQL)
**World Cup 2026 Predictor** · The project data lives in a normalized **MySQL** relational database.
This notebook connects to it and answers football questions with analytical SQL — JOINs, aggregations,
`CASE`, `HAVING`, subqueries and views.

**Schema (5 tables, primary & foreign keys):**
```
teams(team_id PK, team_name, confederation, elo_rating)
matches(match_id PK, match_date, group_name, home_team_id FK, away_team_id FK, venue, home_score, away_score)
predictions(pred_id PK, match_id FK, p_home_win, p_draw, p_away_win, predicted_outcome, pred_home_goals, pred_away_goals)
odds(team_id PK/FK, round_of_32, ..., champion)
players(player_id PK, team_id FK, player_name, position, age, caps, goals, club, is_captain)
```
Build the database first with: `python src/build_database.py`

In [1]:
import pandas as pd
import mysql.connector

cn = mysql.connector.connect(host="localhost", user="root", password="mundial2026", database="worldcup2026")

def q(sql):
    """Run a SQL query and return the result as a DataFrame."""
    cur = cn.cursor()
    cur.execute(sql)
    df = pd.DataFrame(cur.fetchall(), columns=cur.column_names)
    cur.close()
    return df

# Tables and row counts
q("""
SELECT table_name, table_rows
FROM information_schema.tables
WHERE table_schema = 'worldcup2026'
ORDER BY table_name;
""")

,TABLE_NAME,TABLE_ROWS
0,matches,72
1,odds,48
2,players,1246
3,predictions,72
4,teams,48


## 1. Basic JOIN — played matches with team names and confederations
The `matches` table stores team **IDs**; we join `teams` twice (home and away) to get readable names.

In [2]:
q("""
SELECT  m.match_date,
        h.team_name  AS home,
        m.home_score, m.away_score,
        a.team_name  AS away,
        h.confederation AS home_conf,
        a.confederation AS away_conf
FROM        matches m
JOIN  teams h ON m.home_team_id = h.team_id
JOIN  teams a ON m.away_team_id = a.team_id
WHERE m.home_score IS NOT NULL
ORDER BY m.match_date
LIMIT 10;
""")

,match_date,home,home_score,away_score,away,home_conf,away_conf
0,2026-06-11,Mexico,2,0,South Africa,CONCACAF,CAF
1,2026-06-11,South Korea,2,1,Czech Republic,AFC,UEFA
2,2026-06-12,Canada,1,1,Bosnia and Herzegovina,CONCACAF,UEFA
3,2026-06-12,United States,4,1,Paraguay,CONCACAF,CONMEBOL
4,2026-06-13,Qatar,1,1,Switzerland,AFC,UEFA
5,2026-06-13,Brazil,1,1,Morocco,CONMEBOL,CAF
6,2026-06-13,Haiti,0,1,Scotland,CONCACAF,UEFA
7,2026-06-13,Australia,2,0,Turkey,AFC,UEFA
8,2026-06-14,Germany,7,1,Curaçao,UEFA,CONCACAF
9,2026-06-14,Ivory Coast,1,0,Ecuador,CAF,CONMEBOL


## 2. 🌍 Performance by confederation — which continent is doing best?
We turn each played match into points for both teams with `CASE`, `UNION` the home/away perspectives,
then aggregate by confederation. This answers: *which continent is over-performing at the World Cup?*

In [3]:
q("""
WITH team_points AS (
    -- Home perspective
    SELECT h.confederation AS conf,
           CASE WHEN m.home_score > m.away_score THEN 3
                WHEN m.home_score = m.away_score THEN 1 ELSE 0 END AS pts,
           m.home_score AS gf, m.away_score AS ga
    FROM matches m JOIN teams h ON m.home_team_id = h.team_id
    WHERE m.home_score IS NOT NULL
    UNION ALL
    -- Away perspective
    SELECT a.confederation,
           CASE WHEN m.away_score > m.home_score THEN 3
                WHEN m.away_score = m.home_score THEN 1 ELSE 0 END,
           m.away_score, m.home_score
    FROM matches m JOIN teams a ON m.away_team_id = a.team_id
    WHERE m.home_score IS NOT NULL
)
SELECT  conf                              AS confederation,
        COUNT(*)                          AS matches_played,
        SUM(pts)                          AS total_points,
        ROUND(AVG(pts), 2)                AS avg_points,
        ROUND(AVG(gf), 2)                 AS avg_goals_for,
        ROUND(AVG(ga), 2)                 AS avg_goals_against
FROM team_points
GROUP BY conf
ORDER BY avg_points DESC;
""")

,confederation,matches_played,total_points,avg_points,avg_goals_for,avg_goals_against
0,UEFA,16,27,1.69,2.25,1.31
1,CONMEBOL,6,8,1.33,1.50,1.33
2,CONCACAF,6,7,1.17,1.33,1.83
3,AFC,9,10,1.11,1.44,1.89
4,CAF,10,10,1.00,0.70,1.60
5,OFC,1,1,1.00,2.00,2.00


## 3. Aggregation + HAVING — the most experienced and youngest squads
Squad-level stats from the `players` table. `HAVING` filters *after* aggregation.

In [4]:
q("""
SELECT  t.team_name,
        t.confederation,
        ROUND(AVG(p.age), 1)   AS avg_age,
        SUM(p.caps)            AS total_caps,
        SUM(p.goals)           AS total_intl_goals
FROM        players p
JOIN  teams t ON p.team_id = t.team_id
GROUP BY t.team_id
HAVING avg_age IS NOT NULL
ORDER BY avg_age DESC
LIMIT 8;
""")

,team_name,confederation,avg_age,total_caps,total_intl_goals
0,Panama,CONCACAF,30.0,1541,126
1,Iran,AFC,29.8,1167,150
2,Colombia,CONMEBOL,29.6,1042,108
3,Cape Verde,CAF,29.2,803,74
4,Qatar,AFC,28.9,1416,208
5,Brazil,CONMEBOL,28.8,916,152
6,Argentina,CONMEBOL,28.7,1229,220
7,Scotland,UEFA,28.7,973,98


## 4. Subquery — squads more experienced than the tournament average
A subquery computes the global average caps-per-player; the outer query keeps teams above it.

In [5]:
q("""
SELECT  t.team_name,
        ROUND(AVG(p.caps), 1) AS avg_caps
FROM        players p
JOIN  teams t ON p.team_id = t.team_id
GROUP BY t.team_id
HAVING AVG(p.caps) > (SELECT AVG(caps) FROM players)
ORDER BY avg_caps DESC
LIMIT 10;
""")

,team_name,avg_caps
0,Panama,59.3
1,Qatar,54.5
2,Argentina,49.2
3,Mexico,46.7
4,Iran,44.9
5,Croatia,44.6
6,Portugal,43.8
7,Switzerland,43.5
8,Belgium,41.7
9,Jordan,40.9


## 5. The model under the microscope — accuracy via SQL
Join `predictions` with the real `matches` results, derive the actual outcome with `CASE`, and
compute the hit-rate — all in SQL.

In [6]:
q("""
SELECT
    COUNT(*)                                                   AS played,
    SUM(CASE WHEN pred.predicted_outcome =
        CASE WHEN m.home_score > m.away_score THEN 'Home win'
             WHEN m.home_score = m.away_score THEN 'Draw'
             ELSE 'Away win' END
        THEN 1 ELSE 0 END)                                     AS correct_outcomes,
    SUM(CASE WHEN pred.pred_home_goals = m.home_score
              AND pred.pred_away_goals = m.away_score
             THEN 1 ELSE 0 END)                                AS exact_scores
FROM        predictions pred
JOIN  matches m ON pred.match_id = m.match_id
WHERE m.home_score IS NOT NULL;
""")

,played,correct_outcomes,exact_scores
0,24,9,4


## 6. The favourites — title odds joined to confederation
The `odds` table joined to `teams` gives the title race grouped by continent.

In [7]:
q("""
SELECT  t.team_name,
        t.confederation,
        o.champion       AS champion_pct,
        o.semi_final     AS reach_semi_pct
FROM        odds o
JOIN  teams t ON o.team_id = t.team_id
ORDER BY o.champion DESC
LIMIT 10;
""")

,team_name,confederation,champion_pct,reach_semi_pct
0,Argentina,CONMEBOL,20.2,50.0
1,Spain,UEFA,16.8,41.2
2,France,UEFA,13.2,36.3
3,England,UEFA,7.5,26.4
4,Germany,UEFA,4.8,20.6
5,United States,CONCACAF,4.8,19.8
6,Colombia,CONMEBOL,3.8,19.1
7,Brazil,CONMEBOL,3.5,16.2
8,Mexico,CONCACAF,3.3,19.0
9,Portugal,UEFA,3.2,15.1


## 7. A reusable VIEW — a clean, readable results table
Views package a complex join behind a simple name, so the rest of the app (or a BI tool like
Power BI) can query `match_results` without knowing the schema.

In [8]:
cur = cn.cursor()
cur.execute("DROP VIEW IF EXISTS match_results")
cur.execute("""
CREATE VIEW match_results AS
SELECT  m.match_id, m.match_date, m.group_name,
        h.team_name AS home_team, m.home_score,
        m.away_score, a.team_name AS away_team,
        pred.predicted_outcome,
        CASE WHEN m.home_score > m.away_score THEN 'Home win'
             WHEN m.home_score = m.away_score THEN 'Draw'
             WHEN m.home_score IS NULL        THEN NULL
             ELSE 'Away win' END AS actual_outcome
FROM        matches m
JOIN  teams h    ON m.home_team_id = h.team_id
JOIN  teams a    ON m.away_team_id = a.team_id
LEFT JOIN predictions pred ON pred.match_id = m.match_id;
""")
cn.commit(); cur.close()

q("SELECT * FROM match_results WHERE actual_outcome IS NOT NULL ORDER BY match_date DESC LIMIT 8;")

,match_id,match_date,group_name,home_team,home_score,away_score,away_team,predicted_outcome,actual_outcome
0,61,2026-06-17,Group K,Portugal,1,1,DR Congo,Home win,Draw
1,62,2026-06-17,Group K,Uzbekistan,1,3,Colombia,Away win,Away win
2,67,2026-06-17,Group L,England,4,2,Croatia,Away win,Home win
3,68,2026-06-17,Group L,Ghana,1,0,Panama,Away win,Home win
4,49,2026-06-16,Group I,France,3,1,Senegal,Home win,Home win
5,50,2026-06-16,Group I,Iraq,1,4,Norway,Away win,Away win
6,55,2026-06-16,Group J,Argentina,3,0,Algeria,Home win,Home win
7,56,2026-06-16,Group J,Austria,3,1,Jordan,Home win,Home win


## Conclusions

- The project's data now lives in a **normalized MySQL database** (5 tables, primary & foreign keys),
  not just loose CSVs — the missing piece of the *"full Data Science pipeline"*.
- Analytical SQL answers real questions: **CONMEBOL and UEFA lead the confederation table**, the most
  experienced squads, the model's hit-rate, the title race by continent.
- A reusable **VIEW** (`match_results`) exposes a clean results table for BI tools (Power BI) or the app.

**SQL techniques demonstrated:** multi-table JOINs (incl. self-join of `teams`), `CASE`, `GROUP BY` +
`HAVING`, scalar subqueries, CTEs (`WITH`), `UNION ALL`, and a `VIEW`.

`python src/build_database.py` rebuilds the database from the CSVs at any time.

In [9]:
cn.close()